 # Imported all libraries which are required
 
 

In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report,
                              mean_squared_error, mean_absolute_error)
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans


# IMPORTED ALL DATASETS


In [ ]:
matches  = pd.read_csv("matches_1930_2022.csv")
world_cup  = pd.read_csv('world_cup.csv')
ranking22  = pd.read_csv('fifa_ranking_2022-10-06.csv')
ranking26  = pd.read_csv('fifa_ranking_2026-06-08.csv')
schedule26 = pd.read_csv('schedule_2026.csv')

print(f"matches      : {matches.shape}")
print(f"world_cup    : {world_cup.shape}")
print(f"ranking 2022 : {ranking22.shape}")
print(f"ranking 2026 : {ranking26.shape}")
print(f"schedule 2026: {schedule26.shape}")
matches.head(3)


#CREATING A PIPELINE TO RESOLVE PROBLEM OF MULTIPLE NAMES OF SAME COUNTRY

In [ ]:
match_teams = set(matches["home_team"]).union(set(matches["away_team"]))
ranking_teams = set(ranking22["team"])
missing = sorted(match_teams - ranking_teams)
print(missing)

country_name_map = {
    'United States':  'USA',
    'Korea Republic': 'South Korea',
    'IR Iran':  'Iran',
    'China PR':  'China',
    'Côte d\'Ivoire':'Ivory Coast',
    'Republic of Ireland':'Ireland',
    'Bosnia and Herzegovina': 'Bosnia-Herzegovina',
    'North Macedonia':  'Macedonia',
    'Türkiye':  'Turkey',
    "Czech Republic" : "Czech",
    'DR Congo' :  'Congo DR',
    "Czechoslovakia": "Czechia",
    "Dutch East Indies": "Indonesia",
    "FR Yugoslavia": "Serbia",
    "Germany DR": "Germany",
    "Serbia and Montenegro": "Serbia",
    "Soviet Union": "Russia",
    "West Germany": "Germany",
    "Yugoslavia": "Serbia",
    "Zaire": "Congo DR"

}

def standardise_team(name, mapping=country_name_map):
 
    return mapping.get(str(name).strip(), str(name).strip())

for df in [ranking22, ranking26]:
    df['team'] = df['team'].apply(standardise_team)

schedule26['home_team'] = schedule26['home_team'].apply(standardise_team)
schedule26['away_team'] = schedule26['away_team'].apply(standardise_team)



matches['home_team'] = matches['home_team'].apply(standardise_team)
matches['away_team'] = matches['away_team'].apply(standardise_team)

match_teams = set(matches["home_team"]).union(set(matches["away_team"]))
ranking_teams = set(ranking22["team"])
missing = sorted(match_teams - ranking_teams)
print(missing)

print(f"Unique teams in matches: {matches['home_team'].nunique()}")


#Print basic stats and correlation matrix


In [ ]:

print("Basic Statistics")
print(matches[['home_score', 'away_score', 'Year']].describe().round(2))
matches['total_goals'] = matches['home_score'] + matches['away_score']
num_cols = ['home_score', 'away_score', 'total_goals', 'Year']
corr = matches[num_cols].corr()
print("\n Correlation Matrix ")
print(corr.round(3))


# PLOTTING TOP 10 GOAL SCORING TEAMS

In [ ]:

home_goals = matches.groupby('home_team')['home_score'].sum()
away_goals = matches.groupby('away_team')['away_score'].sum()
total_goals_by_country = (home_goals.add(away_goals, fill_value=0)
                           .sort_values(ascending=False)
                           .head(10))

fig, ax = plt.subplots(figsize=(10, 5))
for i in range(0,10) :
  ax.bar(x = i,height = total_goals_by_country.iloc[i])

ax.set_title('Top 10 Countries — Total Historical Goals (FIFA World Cup 1930–2022)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Country')
ax.set_ylabel('Total Goals Scored')
plt.xticks(rotation=45, ha='right')   # prevent X-axis label overlapping
plt.tight_layout()
plt.savefig('top10_goals.png', bbox_inches='tight')
plt.show()


#TARGET VARIABLE ENCODING


In [ ]:
def encode_result(row):
    if row['home_score'] > row['away_score']:
        return 0   
    elif row['home_score'] < row['away_score']:
        return 1  
    else:
        return 2   

matches['Match_Result'] = matches.apply(encode_result, axis=1)

print("Match_Result distribution:")
print(matches['Match_Result'].value_counts()
      .rename({0:'Home Win', 1:'Away Win', 2:'Draw'}))


In [ ]:

matches = matches.sort_values('Year').reset_index(drop=True)

# ── Feature 1 & 2: Historical Win Rate (leak-free expanding window) ──
team_stats = {}
home_wr, away_wr = [], []

for _, row in matches.iterrows():
    ht, at = row['home_team'], row['away_team']
    res = row['Match_Result']

    # Record BEFORE updating (so current match is not included)
    for team in [ht, at]:
        if team not in team_stats:
            team_stats[team] = {'wins': 0, 'games': 0}

    ht_stats = team_stats[ht]
    at_stats = team_stats[at]

    home_wr.append(ht_stats['wins'] / ht_stats['games'] if ht_stats['games'] > 0 else 0)
    away_wr.append(at_stats['wins'] / at_stats['games'] if at_stats['games'] > 0 else 0)

    # NOW update with current match result
    team_stats[ht]['games'] += 1
    team_stats[at]['games'] += 1

    if res == 0:   team_stats[ht]['wins'] += 1
    elif res == 1: team_stats[at]['wins'] += 1

matches['home_hist_win_rate'] = home_wr
matches['away_hist_win_rate'] = away_wr

champ_counts = (world_cup['Champion'].value_counts()
                .rename(index=lambda x: standardise_team(x)).groupby(level=0).sum() )

matches['home_tournament_wins'] = (matches['home_team'].map(champ_counts).fillna(0).astype(int))
matches['away_tournament_wins'] = (matches['away_team'].map(champ_counts).fillna(0).astype(int))

print(matches[['home_team', 'home_hist_win_rate', 'home_tournament_wins',
               'away_team', 'away_hist_win_rate', 'away_tournament_wins']].head(5))


#ENSURING ZERO NULL VALUES IN CRITICAL COLUMNS


In [ ]:
rank_map = ranking22.set_index('team')['rank'].to_dict()

matches['home_rank'] = matches['home_team'].map(rank_map)
matches['away_rank'] = matches['away_team'].map(rank_map)

median_rank = int(np.nanmedian(list(rank_map.values())))
matches['home_rank'] = matches['home_rank'].fillna(median_rank).astype(int)
matches['away_rank'] = matches['away_rank'].fillna(median_rank).astype(int)

print(f"Null home_rank: {matches['home_rank'].isna().sum()}")
print(f"Null away_rank: {matches['away_rank'].isna().sum()}")
print("Zero NaN values in critical rank columns post-merge.")


#TRAIN TEST SPLIT 


In [ ]:
FEATURES = ['home_rank', 'away_rank',
            'home_hist_win_rate', 'away_hist_win_rate',
            'home_tournament_wins', 'away_tournament_wins']
TARGET = 'Match_Result'

train = matches[matches['Year'] <= 2018].copy()
test  = matches[matches['Year'] == 2022].copy()

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f"Train samples : {len(X_train)}")
print(f"Test  samples : {len(X_test)}")
print(f"Class distribution (train):\n{y_train.value_counts().to_string()}")


#LOGISIC REGRESSION MODEL TO PREDICT WHETHER HOME TEAM OR AWAY TEAM WILL WIN 


In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

print("Logistic Regression — Classification Report")
print(classification_report(y_test, lr_preds,
      target_names=['Home Win','Away Win','Draw']))

lr_acc = accuracy_score(y_test, lr_preds)
lr_f1  = f1_score(y_test, lr_preds, average='macro')
print(f"Accuracy : {lr_acc:.4f}")
print(f"Macro F1 : {lr_f1:.4f}")


# Predicting Home Team Wins and Away Team Wins using Forest Classifier


In [ ]:
rf = RandomForestClassifier(n_estimators=300, max_depth=8,
                             random_state=42, class_weight='balanced', n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

print("Random Forest — Classification Report")
print(classification_report(y_test, rf_preds,
      target_names=['Home Win','Away Win','Draw']))

rf_acc = accuracy_score(y_test, rf_preds)
rf_f1  = f1_score(y_test, rf_preds, average='macro')
print(f"Accuracy : {rf_acc:.4f}")
print(f"Macro F1 : {rf_f1:.4f}")


#Comparison Between the two


In [ ]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],

    "Accuracy": [
        lr_acc,
        rf_acc
    ],

    "Macro F1": [
        lr_f1,
        rf_f1
    ],

    "Precision (macro)": [
        precision_score(y_test, lr_preds, average="macro"),
        precision_score(y_test, rf_preds, average="macro")
    ],

    "Recall (macro)": [
        recall_score(y_test, lr_preds, average="macro"),
        recall_score(y_test, rf_preds, average="macro")
    ]
})

comparison = comparison.set_index("Model").round(4)

print(comparison)

comparison[["Accuracy", "Macro F1"]].plot(
    kind="bar",
    figsize=(9,4),
    edgecolor="black"
)

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0,1)

plt.tight_layout()

plt.show()

#Calculating Importance of each Feature used in random forest


In [ ]:
feat_imp = pd.Series(
    rf.feature_importances_,
    index=FEATURES
).sort_values()

plt.figure(figsize=(8,5))

feat_imp.plot(
    kind="barh"
)

plt.title("Random Forest Feature Importance")

plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

#TOTAL GOALS USING RANDOM FOREST REGRESSOR


In [ ]:
TARGET_REG = 'total_goals'

X_train_r, y_train_r = train[FEATURES], train[TARGET_REG]
X_test_r,  y_test_r  = test[FEATURES],  test[TARGET_REG]

rfr = RandomForestRegressor(n_estimators=300, max_depth=6,
                             random_state=42, n_jobs=-1)
rfr.fit(X_train_r, y_train_r)
rfr_preds = rfr.predict(X_test_r)

rmse = np.sqrt(mean_squared_error(y_test_r, rfr_preds))
mae  = mean_absolute_error(y_test_r, rfr_preds)

print(f"Random Forest Regressor (Goal Prediction)")
print(f"  RMSE : {rmse:.4f}")
print(f"  MAE  : {mae:.4f}")

# Scatter: actual vs predicted goals
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test_r, rfr_preds, alpha=0.6, color='steelblue', edgecolors='white')
ax.plot([0, 10], [0, 10], 'r--', linewidth=1.5)
ax.set_xlabel('Actual Total Goals')
ax.set_ylabel('Predicted Total Goals')
ax.set_title('Regression — Actual vs Predicted Goals (2022 WC)', fontweight='bold')
plt.tight_layout()
plt.savefig('regression_goals.png', bbox_inches='tight')
plt.show()


In [ ]:
rank26_map = ranking26.set_index('team')['rank'].to_dict()
median_rank26 = int(np.nanmedian(list(rank26_map.values())))

sch = schedule26[['home_team', 'away_team']].head(10).copy()
sch['home_rank'] = sch['home_team'].map(rank26_map).fillna(median_rank26)
sch['away_rank'] = sch['away_team'].map(rank26_map).fillna(median_rank26)
sch['home_rank'] = sch['home_rank'].astype(float).astype(int)
sch['away_rank'] = sch['away_rank'].astype(float).astype(int)


win_rate_full = {}
for _, row in matches.iterrows():
    ht, at = row['home_team'], row['away_team']
    res = row['Match_Result']
    for team in [ht, at]:
        if team not in win_rate_full:
            win_rate_full[team] = {'wins': 0, 'games': 0}
    win_rate_full[ht]['games'] += 1
    win_rate_full[at]['games'] += 1
    if res == 0:   win_rate_full[ht]['wins'] += 1
    elif res == 1: win_rate_full[at]['wins'] += 1

wr_map_full = {t: v['wins']/v['games'] for t, v in win_rate_full.items() if v['games']}

sch['home_hist_win_rate'] = sch['home_team'].map(wr_map_full).fillna(0)
sch['away_hist_win_rate'] = sch['away_team'].map(wr_map_full).fillna(0)
sch['home_tournament_wins'] = sch['home_team'].map(champ_counts).fillna(0).astype(int)
sch['away_tournament_wins'] = sch['away_team'].map(champ_counts).fillna(0).astype(int)

X_2026 = sch[FEATURES]

proba = lr.predict_proba(X_2026)
forecast = pd.DataFrame({
    'Home Team':    sch['home_team'].values,
    'Away Team':    sch['away_team'].values,
    'Home Win %':   (proba[:, 0] * 100).round(1),
    'Away Win %':   (proba[:, 1] * 100).round(1),
    'Draw %':       (proba[:, 2] * 100).round(1),
})

print("=== 2026 FIFA World Cup — Group Stage Probability Forecast (First 10 Fixtures) ===\n")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print(forecast.to_string(index=False))


#SUMMARY OF PERFORMANCE

In [ ]:

print("FINAL MODEL PERFORMANCE SUMMARY")

print(f"Logistic Regression Accuracy : {lr_acc:.4f}")
print(f"Logistic Regression Macro F1 : {lr_f1:.4f}")

print()

print(f"Random Forest Accuracy : {rf_acc:.4f}")
print(f"Random Forest Macro F1 : {rf_f1:.4f}")

print()
print(forecast.to_string(index=False))

